In [1]:
import os
import json
import numpy as np
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torchvision.models as models

from sklearn.metrics import f1_score
from tqdm import tqdm

In [2]:
DATASET_PATH = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset"

TRAIN_IMG = f"{DATASET_PATH}/train/image"
TRAIN_ANNO = f"{DATASET_PATH}/train/annos"

VAL_IMG = f"{DATASET_PATH}/validation/image"
VAL_ANNO = f"{DATASET_PATH}/validation/annos"

In [3]:
train_files = sorted(os.listdir(TRAIN_IMG))[:30000]
val_files = sorted(os.listdir(VAL_IMG))[:5000]

In [4]:
CLASS_MAP = {1:0,2:1,7:2,8:3,9:4}
NUM_CLASSES = 5

In [5]:
class FashionDataset(Dataset):

    def __init__(self, img_path, anno_path, files, transform=None):
        self.img_path = img_path
        self.anno_path = anno_path
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        img_name = self.files[idx]
        img = np.array(Image.open(os.path.join(self.img_path, img_name)).convert("RGB"))

        anno_file = img_name.replace(".jpg",".json")
        with open(os.path.join(self.anno_path, anno_file)) as f:
            data = json.load(f)

        label = torch.zeros(NUM_CLASSES)

        for item in data.values():
            cid = item["category_id"]
            if cid in CLASS_MAP:
                label[CLASS_MAP[cid]] = 1

        if self.transform:
            img = self.transform(image=img)["image"]

        return img, label

In [6]:
train_transform = A.Compose([
    A.RandomResizedCrop(size=(224,224), scale=(0.8,1.0)),
    A.HorizontalFlip(p=0.5),

    A.OneOf([
        A.ColorJitter(0.3,0.3,0.3,0.1),
        A.RandomBrightnessContrast()
    ], p=0.5),

    A.Affine(translate_percent=0.1, scale=(0.9,1.1), rotate=(-10,10), p=0.5),

    A.CoarseDropout(
        num_holes_range=(1,1),
        hole_height_range=(16,32),
        hole_width_range=(16,32),
        p=0.1
    ),

    A.Normalize(mean=(0.485,0.456,0.406),
                std=(0.229,0.224,0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256,256),
    A.CenterCrop(224,224),
    A.Normalize(mean=(0.485,0.456,0.406),
                std=(0.229,0.224,0.225)),
    ToTensorV2()
])

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = FashionDataset(TRAIN_IMG, TRAIN_ANNO, train_files, train_transform)
val_dataset = FashionDataset(VAL_IMG, VAL_ANNO, val_files, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=(device.type=="cuda"))
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=4, pin_memory=(device.type=="cuda"))

In [8]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 113MB/s] 


In [9]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        loss = (1 - pt) ** self.gamma * bce
        return loss.mean()

criterion = FocalLoss()

In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [11]:
def mixup(images, labels, alpha=0.4):

    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(images.size(0)).to(images.device)

    mixed_images = lam * images + (1 - lam) * images[index]
    labels_a, labels_b = labels, labels[index]

    return mixed_images, labels_a, labels_b, lam

In [12]:
def train_epoch(model, loader):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images, labels = images.to(device), labels.to(device)

        images, labels_a, labels_b, lam = mixup(images, labels)

        optimizer.zero_grad()

        outputs = model(images)

        loss = lam * criterion(outputs, labels_a) + (1-lam) * criterion(outputs, labels_b)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
def validate(model, loader):

    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)
            outputs = model(images)

            preds.append(torch.sigmoid(outputs).cpu())
            targets.append(labels)

    return torch.cat(preds), torch.cat(targets)

In [14]:
def find_best_thresholds(preds, targets):

    preds, targets = preds.numpy(), targets.numpy()
    thresholds = []

    for i in range(preds.shape[1]):
        best_t, best_f1 = 0.5, 0

        for t in np.linspace(0.1,0.9,17):
            f1 = f1_score(targets[:,i], (preds[:,i]>t).astype(int))
            if f1 > best_f1:
                best_f1, best_t = f1, t

        thresholds.append(best_t)

    return thresholds

Now with BCE loss


In [15]:
EPOCHS = 30
best_f1 = 0
patience = 5
counter = 0

for epoch in range(EPOCHS):

    train_loss = train_epoch(model, train_loader)

    preds, targets = validate(model, val_loader)

    thresholds = find_best_thresholds(preds, targets)

    preds_np = preds.numpy()
    targets_np = targets.numpy()

    preds_bin = np.zeros_like(preds_np)

    for i in range(NUM_CLASSES):
        preds_bin[:, i] = (preds_np[:, i] > thresholds[i]).astype(int)

    f1 = f1_score(targets_np, preds_bin, average="macro")

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val F1: {f1:.4f}")
    print(f"Best F1: {best_f1:.4f}")

    # Early Stopping Logic
    if f1 > best_f1:
        best_f1 = f1
        counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print("Best Model saved")
    else:
        counter += 1
        print(f"⚠️ No improvement ({counter}/{patience})")

    if counter >= patience:
        print("Early stopping triggered")
        break


Epoch 1
Train Loss: 0.1285
Val F1: 0.6731
Best F1: 0.0000
Best Model saved

Epoch 2
Train Loss: 0.1160
Val F1: 0.6852
Best F1: 0.6731
Best Model saved

Epoch 3
Train Loss: 0.1092
Val F1: 0.6986
Best F1: 0.6852
Best Model saved

Epoch 4
Train Loss: 0.1070
Val F1: 0.7120
Best F1: 0.6986
Best Model saved

Epoch 5
Train Loss: 0.1003
Val F1: 0.7254
Best F1: 0.7120
Best Model saved

Epoch 6
Train Loss: 0.0986
Val F1: 0.7205
Best F1: 0.7254
⚠️ No improvement (1/5)

Epoch 7
Train Loss: 0.0964
Val F1: 0.7234
Best F1: 0.7254
⚠️ No improvement (2/5)

Epoch 8
Train Loss: 0.0915
Val F1: 0.7194
Best F1: 0.7254
⚠️ No improvement (3/5)

Epoch 9
Train Loss: 0.0916
Val F1: 0.7271
Best F1: 0.7254
Best Model saved

Epoch 10
Train Loss: 0.0913
Val F1: 0.7264
Best F1: 0.7271
⚠️ No improvement (1/5)

Epoch 11
Train Loss: 0.0877
Val F1: 0.7258
Best F1: 0.7271
⚠️ No improvement (2/5)

Epoch 12
Train Loss: 0.0856
Val F1: 0.7222
Best F1: 0.7271
⚠️ No improvement (3/5)

Epoch 13
Train Loss: 0.0856
Val F1: 0.7226

Now with BCE Loss

In [23]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)

model = model.to(device)

In [24]:
from collections import defaultdict

class_counts = defaultdict(int)

for img in train_files:
    anno = img.replace(".jpg", ".json")
    anno_path = os.path.join(TRAIN_ANNO, anno)

    with open(anno_path) as f:
        data = json.load(f)

    for item in data.values():
        cid = item["category_id"]
        if cid in CLASS_MAP:
            class_counts[cid] += 1

print("Class counts:", class_counts)

#  Compute weights
total = sum(class_counts.values())

weights = []
for cid in [1,2,7,8,9]:
    w = total / (NUM_CLASSES * class_counts[cid])
    weights.append(w)

weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Class weights:", weights)

Class counts: defaultdict(<class 'int'>, {8: 11710, 1: 14796, 2: 7435, 9: 6163, 7: 7648})
Class weights: tensor([0.6455, 1.2845, 1.2487, 0.8156, 1.5496], device='cuda:0')


In [43]:
criterion = nn.BCEWithLogitsLoss(pos_weight=weights)

In [44]:
train_transform = A.Compose([
   
    A.RandomResizedCrop(size=(256,256), scale=(0.75,1.0)),

    #  Geometry
    A.HorizontalFlip(p=0.5),
    A.Affine(
        translate_percent=0.08,
        scale=(0.9,1.1),
        rotate=(-10,10),
        p=0.6
    ),

    #  Color robustness
    A.OneOf([
        A.ColorJitter(0.25,0.25,0.25,0.05),
        A.RandomBrightnessContrast(),
    ], p=0.5),

    #  Blur / noise (real-world effect)
    A.OneOf([
        A.MotionBlur(blur_limit=3),
        A.GaussianBlur(blur_limit=3),
        A.GaussNoise(std_range=(0.1, 0.3)),
    ], p=0.3),

    #  Mild occlusion (reduced strength)
    A.CoarseDropout(
        num_holes_range=(1,1),
        hole_height_range=(12,24),
        hole_width_range=(12,24),
        p=0.1
    ),

    #  Normalize
    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()
])

In [45]:
val_transform = A.Compose([
    A.Resize(288,288),
    A.CenterCrop(256,256),

    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()
])

In [47]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [48]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,   # total epochs
    eta_min=1e-6    # minimum LR
)

In [49]:
def train_epoch(model, loader):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [50]:
def validate(model, loader):

    model.eval()
    total_loss = 0

    preds = []
    targets = []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            #  compute validation loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            predictions = torch.sigmoid(outputs)

            preds.append(predictions.cpu())
            targets.append(labels.cpu())

    preds = torch.cat(preds)
    targets = torch.cat(targets)

    avg_loss = total_loss / len(loader)

    return avg_loss, preds, targets

In [52]:
EPOCHS = 30
best_f1 = 0
patience = 5
counter = 0

for epoch in range(EPOCHS):

    train_loss = train_epoch(model, train_loader)

    val_loss, preds, targets = validate(model, val_loader)

    thresholds = find_best_thresholds(preds, targets)

    preds_np = preds.numpy()
    targets_np = targets.numpy()

    preds_bin = np.zeros_like(preds_np)

    for i in range(NUM_CLASSES):
        preds_bin[:, i] = (preds_np[:, i] > thresholds[i]).astype(int)

    f1 = f1_score(targets_np, preds_bin, average="macro")
    scheduler.step()

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val F1: {f1:.4f}")

    # Early Stopping Logic
    if f1 > best_f1:
        best_f1 = f1
        counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model_BCE.pth")
        print("Best Model saved")
    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})")

    if counter >= patience:
        print("Early stopping triggered")
        break


Epoch 1
Train Loss: 0.1619
Val Loss: 0.4196
Val F1: 0.7231
Best Model saved

Epoch 2
Train Loss: 0.1497
Val Loss: 0.4631
Val F1: 0.7216
No improvement (1/5)

Epoch 3
Train Loss: 0.1367
Val Loss: 0.4679
Val F1: 0.7295
Best Model saved

Epoch 4
Train Loss: 0.1241
Val Loss: 0.4927
Val F1: 0.7189
No improvement (1/5)

Epoch 5
Train Loss: 0.1185
Val Loss: 0.5330
Val F1: 0.7252
No improvement (2/5)

Epoch 6
Train Loss: 0.1075
Val Loss: 0.4861
Val F1: 0.7269
No improvement (3/5)

Epoch 7
Train Loss: 0.1007
Val Loss: 0.5076
Val F1: 0.7270
No improvement (4/5)

Epoch 8
Train Loss: 0.0918
Val Loss: 0.5503
Val F1: 0.7223
No improvement (5/5)
Early stopping triggered
